# 01 - Build Data Layers

Run the data-build scripts in order. Each cell calls one CLI script via `uv run`.

0. **Manifest check** — verify all downloaded raw VD dates are complete; list any gaps
1. station 1m / 5m — date range **2023-06-01..2025-05-31** (requires raw VD for 2023-06..2024-03 to be pre-downloaded locally)
2. historical speed profile (DuckDB)
3. per-accident samples (NPZ + per-accident CSV) — date range **2023-06-01..2025-05-31**, always rebuilt with `--force`
4. 33 incident features

Each build step is idempotent unless `--force` is passed.

In [ ]:
import os, sys

# ── environment detection ────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Google Drive mount (Colab only) ──────────────────────────────────────────
# force_remount=False (default): skips re-auth if already mounted in this runtime.
# To avoid OAuth on every new session, store a service-account key in
# Colab Secrets (left sidebar → 🔑) as "GDRIVE_SA_KEY" and uncomment below:
#
#   from google.colab import userdata
#   import json, tempfile, google.auth
#   from google.oauth2 import service_account
#   _key = json.loads(userdata.get('GDRIVE_SA_KEY'))
#   _creds = service_account.Credentials.from_service_account_info(
#       _key, scopes=['https://www.googleapis.com/auth/drive'])
#   # then use gcsfs / PyDrive2 with _creds instead of drive.mount()

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DEFAULT_ROOT = '/content/drive/MyDrive/陽交大高速公路計畫/main'
else:
    # Local (VSCode/Jupyter): walk up if notebook lives in <root>/notebooks/
    _here = os.path.abspath(os.getcwd())
    _DEFAULT_ROOT = os.path.dirname(_here) if os.path.basename(_here) == 'notebooks' else _here

PROJECT_ROOT = os.environ.get('PROJECT_ROOT', _DEFAULT_ROOT)
%cd {PROJECT_ROOT}
print('PROJECT_ROOT =', PROJECT_ROOT)
!pwd

## 0. Manifest check

Scan every manifest JSON in `data/_vd_download_manifests/` and report:
- complete vs incomplete dates
- dates that have a complete manifest but no corresponding parquet on Drive
- the full date range covered

Incomplete dates must be re-downloaded **locally** via `download_vd_drive.py` before the station build can use them.

In [ ]:
import json
from pathlib import Path

data_root = Path(PROJECT_ROOT) / 'data'
manifest_dir = data_root / '_vd_download_manifests'
raw_dir      = data_root / 'vd_raw_lane_1m_full'

TOTAL_MINUTES = 1440  # 1 per minute * 24h

manifest_files = sorted(manifest_dir.rglob('*.manifest.json'))
print(f'Manifest files found: {len(manifest_files)}')

complete, incomplete, errored = [], [], []

for mf in manifest_files:
    try:
        d = json.loads(mf.read_text(encoding='utf-8'))
    except Exception as e:
        errored.append((mf.name, str(e)))
        continue

    date_str   = d.get('date', '')
    counts     = d.get('counts', {})
    downloaded = counts.get('downloaded', 0)
    s404       = counts.get('server_404', 0)
    is_complete = d.get('complete', (downloaded + s404 == TOTAL_MINUTES))

    entry = dict(date=date_str, downloaded=downloaded, server_404=s404,
                 http_error=counts.get('http_error', 0),
                 parse_error=counts.get('parse_error', 0))
    (complete if is_complete else incomplete).append(entry)

print(f'\nComplete  : {len(complete):>4} days')
print(f'Incomplete: {len(incomplete):>4} days')
print(f'Errored   : {len(errored):>4} manifest files')

if complete:
    dates = sorted(e['date'] for e in complete)
    print(f'\nCovered range: {dates[0]}  →  {dates[-1]}')

if incomplete:
    print(f'\nIncomplete dates (up to 30 shown):')
    for e in incomplete[:30]:
        print(f"  {e['date']}  dl={e['downloaded']:4}  404={e['server_404']:4}  "
              f"http_err={e['http_error']}  parse_err={e['parse_error']}")
    if len(incomplete) > 30:
        print(f'  ... and {len(incomplete)-30} more')

# cross-check: complete manifests that have no parquet on Drive
raw_stems = {f.stem for f in raw_dir.rglob('vd_*.parquet')}  # e.g. 'vd_20240401'
missing_parquet = [
    e['date'] for e in complete
    if f"vd_{e['date'].replace('-', '')}" not in raw_stems
]
if missing_parquet:
    print(f'\nWARNING: {len(missing_parquet)} complete-manifest dates have no parquet on Drive:')
    for d in sorted(missing_parquet)[:20]:
        print(f'  {d}')
else:
    print(f'\nAll {len(complete)} complete dates have a matching parquet on Drive. ✓')

## 1. Station 1m / 5m

Builds `vd_station_1m` and `vd_station_5m` from raw lane-level parquet.
Skips dates whose output already exists; use `--force` to overwrite.

**Date range extended to 2023-06-01.**  
Dates before 2024-04-01 are silently skipped if the corresponding raw parquet is not yet on Drive.
To fill the gap, run `download_vd_drive.py --start-date 2023-06-01 --end-date 2024-03-31` locally first.

In [ ]:
!uv run scripts/data/build_station_vd_drive.py --start-date 2023-06-01 --end-date 2025-05-31

## 2. Historical speed profile (DuckDB)

Aggregates the full `vd_station_5m` dataset into weekday / holiday speed profiles per VD.
The `--start-date` here only affects the DuckDB scan window — it reads whatever parquet is on Drive.
The range 2021-06-01 is deliberately wide so that future earlier data is included automatically.

In [ ]:
!uv run scripts/data/build_historical_profile.py --start-date 2021-06-01 --end-date 2025-05-31 --force

## 3. Per-accident sample matrices (NPZ + per-accident CSV)

Default window: 50 km upstream + 10 km downstream, −60 to +240 min.
The NPZ stack is the model input; CSV / JSON sidecar files are for debug and catalog use.

**Date range extended to 2023-06-01** to include accidents from the SMS records era once that data is integrated into `accident_data.csv`.

`--force` is always passed so every accident is re-evaluated against the current station data
(picks up any newly downloaded 2023 VD, fills gaps where `filled_ratio` was low).

In [ ]:
!uv run scripts/data/build_accident_samples.py --start-date 2023-06-01 --end-date 2025-05-31 --force

## 3b. Per-accident historical-baseline matrices (for VD+History models)

Builds `data/historical_matrices_5m.npz`, one historical-baseline matrix per accident aligned
to the sample NPZ. Required by the `--input-type vd+hist` / `hist` training runs.

In [ ]:
!uv run scripts/data/build_historical_matrices.py

## 4. 33 incident features

Reads `data/accident_data.csv` and geometry CSVs; outputs a 33-column feature table.
Weather features fall back to zeros if no `--weather-csv` is provided.

In [ ]:
!uv run scripts/features/build_incident_features.py

## Final sanity check

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path(PROJECT_ROOT)))
from scripts.lib.data_loaders import load_samples
samples = load_samples()
print(f'samples: {len(samples)}, matrix shape={samples.matrix.shape}, window={samples.window.to_dict()}')
print(samples.metadata[['accident_id','freeway','direction','filled_ratio','status']].head().to_string())